In [1]:
GROQ_API_KEY = "Enter your secret key here."

In [ ]:
!pip install groq

In [2]:
# Prompt => openai model => response

In [3]:
from groq import Groq
import os

client = Groq(api_key=GROQ_API_KEY)

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",  # active model ID
    messages=[
        {"role": "user", "content": "a+2=7, find a"}
    ]
)

print(response.choices[0].message.content)

\(a = 5\)


In [4]:
# Image
response = client.chat.completions.create(
    model="meta-llama/llama-4-scout-17b-16e-instruct",  #  active vision model
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text", 
                    "text": "tell me the cpital of the country to which the flag belongs to in the reference image"
                },
                {
                    "type": "image_url", 
                    "image_url": {
                        "url": "https://vipschool.in/wp-content/uploads/2023/08/Picture1-3.png"
                    }
                }
            ]
        }
    ]
)

print(response.choices[0].message.content)

The flag in the reference image is the flag of India. The capital of India is New Delhi.


### AI Personal Assistant 

In [5]:
# queries => answer
#emails => summarize

class PersonalAssistant:
    def __init__(self):
        print("Hi, I am your AI assistant, How can I help you?")
    def ans_query(self):
        question = input("Ask me anything")
        
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",  # active model ID
            messages=[
                {"role": "system", "content": "act like a helpful personal assistant"},
                {"role": "user", "content": question}
            ],
            temperature=0.7,
            max_completion_tokens=512
        )

        
        print(response.choices[0].message.content.strip())

    def summarize_email(self):
        email_text = input("Paste your email here: ")
        prompt = f"Summarize the following email in 2-3 sentences: {email_text}"
    
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[
                {"role": "system", "content": "act like an expert email assistant"},
                {"role": "user", "content": prompt}  # Changed 'question' to 'prompt'
        ],
        temperature=0.3,
        max_completion_tokens=512
    )
    
        print("\n\nSummary: ",response.choices[0].message.content.strip())
        

In [6]:
assistant = PersonalAssistant()

Hi, I am your AI assistant, How can I help you?


In [8]:
assistant.ans_query()

Ask me anything hi


Hey there! How can I help you today?


In [7]:
assistant.summarize_email()

Paste your email here:  hi




Summary:  The email consists solely of a brief greeting: “hi.” It is concise and provides no additional context or information beyond the simple salutation.


In [8]:
#Sample email

# """Respected Rajesh sir,
# MAhee Tailor here ,I hope you are doing well.
# I am writing to formally request a review of my
# current compensation. I have thoroughly enjoyed 
# working at sarla over the past , and I am proud 
# of the milestones our team has achieved during 
# this time.Since my last salary review, I have 
# consistently focused on delivering high-quality
# work and expanding my contributions to the company."""

### Images in Groq

#### Responses API

In [9]:
import base64
# images => base64 encoded strings => decode

In [18]:
import base64
import json
import requests  # Make sure to run: pip install requests

# 1. Your existing completion code
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user", 
            "content": "generate an image of a highly futuristic spaceship, flying near a nebula"
        }
    ],
    tools=[
        {
            "type": "function",
            "function": {
                "name": "generate_image",
                "description": "Call this tool to generate an image based on a detailed text prompt.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "prompt": {
                            "type": "string",
                            "description": "The detailed visual description for the image generation."
                        }
                    },
                    "required": ["prompt"]
                }
            }
        }
    ]
)

# Check if the model decided to call your function
tool_calls = response.choices[0].message.tool_calls

if tool_calls:
    # Extract the prompt from the model's tool call arguments
    tool_arguments = json.loads(tool_calls[0].function.arguments)
    prompt_text = tool_arguments.get("prompt")
    print(f"Generating image for prompt: {prompt_text}\n")
    
    # --- NEW: ACTUALLY GENERATE THE IMAGE ---
    # Encode prompt for the URL
    from urllib.parse import quote
    encoded_prompt = quote(prompt_text)
    image_url = f"https://image.pollinations.ai/p/{encoded_prompt}?width=1024&height=1024&seed=42"
    
    # Download the image data from the provider
    img_response = requests.get(image_url)
    
    if img_response.status_code == 200:
        # Save the bytes directly as a PNG file
        with open("spaceship.png", "wb") as f:
            f.write(img_response.content)
        print("Success! 'spaceship.png' has been saved to your workspace.")
    else:
        print("Failed to download the generated image.")

Generating image for prompt: A highly futuristic spaceship soaring through space, its sleek, aerodynamic hull gleaming with reflective panels and iridescent colors. The vessel is adorned with advanced propulsion thrusters, emitting a subtle glow, and it drifts gracefully near a vibrant, swirling nebula. The nebula is a tapestry of deep purple, electric blue, and soft pink clouds, illuminated by distant stars. The background is a star-filled cosmos, with distant galaxies and a faint Milky Way. The scene captures a sense of speed, elegance, and the awe of interstellar travel.

Success! 'spaceship.png' has been saved to your workspace.


#### Image API

In [23]:
import requests
from urllib.parse import quote

# 1. Groq can't do images, so we use a free, direct Image API 
prompt = "A highly futuristic spaceship flying near a nebula"
encoded_prompt = quote(prompt)
image_url = f"https://image.pollinations.ai/p/{encoded_prompt}?width=1024&height=1024&seed=42"

print(f"Generating image for: '{prompt}'...")
img_response = requests.get(image_url)

if img_response.status_code == 200:
    # 2. Extract the raw bytes (similar to image_bytes from your old code)
    image_bytes = img_response.content
    
    # 3. Save the image to a file
    with open("spaceship_API.png", "wb") as f:
        f.write(image_bytes)
        
    print("Success! 'spaceship_API.png' has been saved.")
else:
    print(f"Failed to generate image. Status code: {img_response.status_code}")

Generating image for: 'A highly futuristic spaceship flying near a nebula'...
Success! 'spaceship_API.png' has been saved.


In [24]:
# EDIT
import requests
from urllib.parse import quote

# 1. Groq can't do images, so we use a free, direct Image API 
prompt = "take the reference image and convert the spaceship color to red."
encoded_prompt = quote(prompt)
image_url = f"https://image.pollinations.ai/p/{encoded_prompt}?width=1024&height=1024&seed=42"
messages = [
    open("spaceship_API.png", "rb")
]

print(f"Generating image for: '{prompt}'...")
img_response = requests.get(image_url)

if img_response.status_code == 200:
    # 2. Extract the raw bytes (similar to image_bytes from your old code)
    image_bytes = img_response.content
    
    # 3. Save the image to a file
    with open("spaceship-red.png", "wb") as f:
        f.write(image_bytes)
        
    print("Success! 'spaceship_red.png' has been saved.")
else:
    print(f"Failed to generate image. Status code: {img_response.status_code}")


Generating image for: 'take the reference image and convert the spaceship color to red.'...
Success! 'spaceship_red.png' has been saved.
